# HertsBike: synthetic SQLite database generator

In [1]:
import os
import math
import sqlite3
from datetime import date, datetime, timedelta

import numpy as np
import pandas as pd

SEED = 2026
rng = np.random.default_rng(SEED)

DB_PATH = "hertsbike.db"

START = date(2025, 11, 1)
END = date(2026, 10, 31)
N_DAYS = (END - START).days + 1

N_BASE_MEMBERS = 1400
N_BIKES = 320
N_TRIPS = 25000

## 1. Deliberate imperfections

In [2]:
RATES = {
    "email_missing": 0.06,        # optional at sign-up
    "phone_missing": 0.12,        # optional
    "birth_year_missing": 0.05,   # optional
    "member_duplicate": 0.04,     # forgot login, registered again
    "rating_missing": 0.40,       # prompt dismissed
    "distance_missing": 0.03,     # GPS dropouts
    "trip_duplicate": 0.015,      # app re-upload on reconnect
    "rainfall_missing": 0.02,     # gauge outages
    "cost_missing": 0.04,         # invoice not matched yet
}

## 2. Reference lists

In [3]:
FIRST_NAMES = [
    "Oliver", "George", "Harry", "Noah", "Jack", "Leo", "Oscar", "Charlie", "Archie", "Freddie",
    "Thomas", "James", "William", "Henry", "Alfie", "Joshua", "Daniel", "Samuel", "Ethan", "Mohammed",
    "Arjun", "Rohan", "Aarav", "Imran", "Tariq", "Kwame", "Chidi", "Tomasz", "Piotr", "Luca",
    "Callum", "Rhys", "Connor", "Declan", "Ryan", "Jamie", "Liam", "Adam", "Ben", "Matthew",
    "Olivia", "Amelia", "Isla", "Ava", "Emily", "Sophie", "Grace", "Lily", "Mia", "Ella",
    "Charlotte", "Jessica", "Hannah", "Chloe", "Megan", "Holly", "Poppy", "Freya", "Ruby", "Evie",
    "Priya", "Ananya", "Aisha", "Fatima", "Zara", "Meera", "Nadia", "Amara", "Folake", "Ngozi",
    "Katarzyna", "Agnieszka", "Elena", "Sofia", "Maria", "Laura", "Rebecca", "Katie", "Lucy", "Emma",
    "Sarah", "Rachel", "Natalie", "Georgia", "Abigail", "Imogen", "Niamh", "Siobhan", "Orla", "Alice",
]
LAST_NAMES = [
    "Smith", "Jones", "Taylor", "Brown", "Williams", "Wilson", "Johnson", "Davies", "Robinson", "Wright",
    "Thompson", "Evans", "Walker", "White", "Roberts", "Green", "Hall", "Wood", "Jackson", "Clarke",
    "Patel", "Singh", "Khan", "Ahmed", "Hussain", "Ali", "Begum", "Sharma", "Kumar", "Gupta",
    "Shah", "Mistry", "Desai", "Joshi", "Reddy", "Nair", "Chowdhury", "Rahman", "Malik", "Iqbal",
    "Okafor", "Adeyemi", "Mensah", "Owusu", "Nwosu", "Balogh", "Kowalski", "Nowak", "Wisniewski", "Rossi",
    "Murphy", "Kelly", "Byrne", "Walsh", "O'Brien", "Ryan", "Doyle", "McCarthy", "Gallagher", "Kennedy",
    "Campbell", "Stewart", "Murray", "Robertson", "MacDonald", "Fraser", "Graham", "Hamilton", "Ross", "Reid",
    "Hughes", "Morgan", "Edwards", "Griffiths", "Lloyd", "Owen", "Price", "Powell", "Rees", "Jenkins",
    "Cooper", "Harrison", "Ward", "Turner", "Parker", "Mitchell", "Bennett", "Carter", "Baker", "Collins",
    "Foster", "Hunt", "Palmer", "Shaw", "Bailey", "Cox", "Webb", "Fisher", "Marshall", "Gibson",
    "Ellis", "Barnes", "Lawrence", "Chapman", "Mills", "Knight", "Stevens", "Holmes", "Newman", "Gardner",
]

# Town centres (approximate public coordinates) and rough populations.
TOWNS = {
    "Hatfield":            (51.7630, -0.2230, 41000),
    "St Albans":           (51.7550, -0.3360, 82000),
    "Welwyn Garden City":  (51.8010, -0.2080, 48000),
    "Hemel Hempstead":     (51.7530, -0.4490, 95000),
    "Watford":             (51.6560, -0.3960, 102000),
    "Stevenage":           (51.9020, -0.2020, 87000),
    "Hertford":            (51.7960, -0.0780, 27000),
    "Harpenden":           (51.8170, -0.3570, 30000),
}
TOWN_NAMES = list(TOWNS)

# Typical dock capacity by site type: stations and campuses are busier than park gates.
SITE_CAPACITY = {
    "Railway Station": 32, "Town Centre": 28, "University Campus": 28, "Bus Station": 24,
    "Retail Park": 20, "Hospital": 16, "Leisure Centre": 16, "Library": 12, "Park Gate": 12,
}

## 3. Schema

Composite keys: `docks`, `weather_daily` and `maintenance_log`. `trips` has two composite foreign keys into `docks`.

Measurement levels: nominal (town, bike_type, service_type, dock_type, preferred_payment),
ordinal (membership_tier, rating, inspection_score), interval (avg_temp_c, birth_year, all dates and times),
ratio (duration_min, distance_km, fare_gbp, rainfall_mm, cost_gbp, purchase_price_gbp, capacity, battery_wh).

In [4]:
SCHEMA = """
PRAGMA foreign_keys = ON;

CREATE TABLE towns (
    town_name   TEXT PRIMARY KEY,
    centre_lat  REAL    NOT NULL CHECK (centre_lat BETWEEN 51.0 AND 52.5),
    centre_lon  REAL    NOT NULL CHECK (centre_lon BETWEEN -1.0 AND 0.5),
    population  INTEGER NOT NULL CHECK (population > 0)
);

CREATE TABLE stations (
    station_id   INTEGER PRIMARY KEY,
    station_name TEXT    NOT NULL UNIQUE,
    town         TEXT    NOT NULL REFERENCES towns(town_name),
    latitude     REAL    NOT NULL CHECK (latitude  BETWEEN 51.0 AND 52.5),
    longitude    REAL    NOT NULL CHECK (longitude BETWEEN -1.0 AND 0.5),
    capacity     INTEGER NOT NULL CHECK (capacity BETWEEN 8 AND 40),
    install_date TEXT    NOT NULL
);

CREATE TABLE docks (
    station_id INTEGER NOT NULL REFERENCES stations(station_id),
    dock_no    INTEGER NOT NULL CHECK (dock_no >= 1),
    dock_type  TEXT    NOT NULL CHECK (dock_type IN ('Standard', 'Charging')),
    PRIMARY KEY (station_id, dock_no)
);

CREATE TABLE bikes (
    bike_id            INTEGER PRIMARY KEY,
    bike_type          TEXT    NOT NULL CHECK (bike_type IN ('Standard', 'Electric', 'Cargo')),
    model              TEXT    NOT NULL,
    purchase_date      TEXT    NOT NULL,
    purchase_price_gbp REAL    NOT NULL CHECK (purchase_price_gbp > 0),
    battery_wh         INTEGER CHECK (battery_wh IS NULL OR battery_wh BETWEEN 200 AND 800),
    -- only electric bikes have a battery, and every electric bike has one
    CHECK ((bike_type = 'Electric') = (battery_wh IS NOT NULL))
);

-- email is deliberately NOT unique: the sign-up system did not de-duplicate, which is how duplicate people arise.
CREATE TABLE members (
    member_id         INTEGER PRIMARY KEY,
    full_name         TEXT    NOT NULL,
    email             TEXT    CHECK (email IS NULL OR email LIKE '%_@_%._%'),
    phone             TEXT,
    birth_year        INTEGER CHECK (birth_year IS NULL OR birth_year BETWEEN 1930 AND 2010),
    home_town         TEXT    NOT NULL REFERENCES towns(town_name),
    membership_tier   TEXT    NOT NULL CHECK (membership_tier IN ('Casual', 'Standard', 'Premium')),
    preferred_payment TEXT    NOT NULL CHECK (preferred_payment IN
                              ('Debit card', 'Credit card', 'Apple Pay', 'Google Pay', 'Direct debit')),
    join_date         TEXT    NOT NULL
);

CREATE TABLE weather_daily (
    obs_date    TEXT NOT NULL,
    town        TEXT NOT NULL REFERENCES towns(town_name),
    avg_temp_c  REAL NOT NULL CHECK (avg_temp_c BETWEEN -25 AND 45),
    rainfall_mm REAL CHECK (rainfall_mm IS NULL OR rainfall_mm BETWEEN 0 AND 300),
    PRIMARY KEY (obs_date, town)
);

CREATE TABLE maintenance_log (
    bike_id          INTEGER NOT NULL REFERENCES bikes(bike_id),
    service_date     TEXT    NOT NULL,
    service_type     TEXT    NOT NULL CHECK (service_type IN
                     ('Routine check', 'Brake service', 'Tyre replacement', 'Chain and drive',
                      'Battery service', 'Damage repair')),
    inspection_score INTEGER NOT NULL CHECK (inspection_score BETWEEN 1 AND 5),
    cost_gbp         REAL    CHECK (cost_gbp IS NULL OR cost_gbp >= 0),
    PRIMARY KEY (bike_id, service_date)
);

-- trip_id is a surrogate key, so exact re-uploads of a trip get a new id and survive as duplicate rows.
CREATE TABLE trips (
    trip_id          INTEGER PRIMARY KEY,
    member_id        INTEGER NOT NULL REFERENCES members(member_id),
    bike_id          INTEGER NOT NULL REFERENCES bikes(bike_id),
    start_station_id INTEGER NOT NULL,
    start_dock_no    INTEGER NOT NULL,
    end_station_id   INTEGER NOT NULL,
    end_dock_no      INTEGER NOT NULL,
    start_time       TEXT    NOT NULL,
    end_time         TEXT    NOT NULL,
    duration_min     INTEGER NOT NULL CHECK (duration_min BETWEEN 1 AND 1440),
    distance_km      REAL    CHECK (distance_km IS NULL OR distance_km BETWEEN 0 AND 200),
    fare_gbp         REAL    NOT NULL CHECK (fare_gbp >= 0),
    rating           INTEGER CHECK (rating IS NULL OR rating BETWEEN 1 AND 5),
    CHECK (end_time > start_time),
    FOREIGN KEY (start_station_id, start_dock_no) REFERENCES docks(station_id, dock_no),
    FOREIGN KEY (end_station_id,   end_dock_no)   REFERENCES docks(station_id, dock_no)
);

CREATE INDEX idx_trips_member ON trips(member_id);
CREATE INDEX idx_trips_bike   ON trips(bike_id);
CREATE INDEX idx_trips_start  ON trips(start_time);

CREATE VIEW v_trip_weather AS
SELECT t.trip_id, t.start_time, t.duration_min, t.distance_km, t.fare_gbp, t.rating,
       s.town, w.avg_temp_c, w.rainfall_mm
FROM trips t
JOIN stations s      ON s.station_id = t.start_station_id
JOIN weather_daily w ON w.town = s.town AND w.obs_date = substr(t.start_time, 1, 10);
"""

if os.path.exists(DB_PATH):
    os.remove(DB_PATH)
con = sqlite3.connect(DB_PATH)
con.executescript(SCHEMA)
con.commit()
print("schema created")

schema created


## 4. Generate the tables

### 4.1 Towns, stations and docks

In [5]:
def haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in km. Used so trip distance depends on where the stations really are."""
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi = p2 - p1
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlmb / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


towns_rows = [(t, lat, lon, pop) for t, (lat, lon, pop) in TOWNS.items()]

stations_rows, docks_rows = [], []
station_town, station_cap, station_lat, station_lon = [], [], [], []
sid = 0
for town, (clat, clon, _) in TOWNS.items():
    sites = rng.choice(list(SITE_CAPACITY), size=5, replace=False)
    for site in sites:
        sid += 1
        cap = int(SITE_CAPACITY[site] + rng.choice([-4, 0, 0, 4]))
        lat = round(clat + rng.normal(0, 0.008), 5)
        lon = round(clon + rng.normal(0, 0.012), 5)
        installed = date(2023, 1, 1) + timedelta(days=int(rng.integers(0, 900)))
        stations_rows.append((sid, f"{town} {site}", town, lat, lon, cap, installed.isoformat()))
        station_town.append(town)
        station_cap.append(cap)
        station_lat.append(lat)
        station_lon.append(lon)
        n_charge = cap // 4
        for d in range(1, cap + 1):
            docks_rows.append((sid, d, "Charging" if d <= n_charge else "Standard"))

n_stations = len(stations_rows)
station_cap = np.array(station_cap)
station_town_idx = np.array([TOWN_NAMES.index(t) for t in station_town])

# Popularity: bigger stations are busier, with a log-normal wobble so no two are alike.
station_pop = station_cap * rng.lognormal(0, 0.35, size=n_stations)
station_pop = station_pop / station_pop.sum()

print(len(towns_rows), "towns,", n_stations, "stations,", len(docks_rows), "docks")

8 towns, 40 stations, 836 docks


### 4.2 Weather

In [6]:
days = [START + timedelta(days=i) for i in range(N_DAYS)]
doy = np.array([d.timetuple().tm_yday for d in days])

seasonal = 10.8 + 7.2 * np.sin(2 * np.pi * (doy - 110) / 365)
anom = np.zeros(N_DAYS)
for i in range(1, N_DAYS):
    anom[i] = 0.7 * anom[i - 1] + rng.normal(0, 2.2)

regional_wet = rng.random(N_DAYS) < 0.38
town_offset = rng.normal(0, 0.4, size=len(TOWN_NAMES))

temp = np.zeros((len(TOWN_NAMES), N_DAYS))
rain = np.zeros((len(TOWN_NAMES), N_DAYS))
for ti in range(len(TOWN_NAMES)):
    temp[ti] = np.round(seasonal + anom + town_offset[ti] + rng.normal(0, 0.7, N_DAYS), 1)
    wet = np.where(regional_wet, rng.random(N_DAYS) < 0.9, rng.random(N_DAYS) < 0.05)
    rain[ti] = np.where(wet, np.round(rng.gamma(0.8, 5.0, N_DAYS), 1), 0.0)

weather_full = []
for ti, town in enumerate(TOWN_NAMES):
    for di, d in enumerate(days):
        weather_full.append((d.isoformat(), town, float(temp[ti, di]), float(rain[ti, di])))
print(len(weather_full), "weather rows")

2920 weather rows


### 4.3 Bikes and maintenance

In [7]:
BIKE_SPECS = {
    # type: (probability, models, price mean, price sd)
    "Standard": (0.60, ["HB Urban 7", "HB Urban 3"], 780, 40),
    "Electric": (0.35, ["HB Volt E2", "HB Volt E3"], 1650, 90),
    "Cargo":    (0.05, ["HB Haul C1"], 2400, 120),
}
bike_types = rng.choice(list(BIKE_SPECS), size=N_BIKES, p=[v[0] for v in BIKE_SPECS.values()])

bikes_rows = []
bike_purchase = []
for i, bt in enumerate(bike_types, start=1):
    _, models, pm, psd = BIKE_SPECS[bt]
    bought = date(2022, 6, 1) + timedelta(days=int(rng.integers(0, 1490)))
    price = round(float(rng.normal(pm, psd)), 2)
    battery = int(rng.choice([400, 500])) if bt == "Electric" else None      # structural NULL for non-electric
    bikes_rows.append((i, bt, str(rng.choice(models)), bought.isoformat(), price, battery))
    bike_purchase.append(bought)

SERVICE_TYPES = ["Routine check", "Brake service", "Tyre replacement", "Chain and drive", "Damage repair", "Battery service"]
SERVICE_P = np.array([0.45, 0.15, 0.15, 0.10, 0.05, 0.10])
SERVICE_COST_MEDIAN = {"Routine check": 18, "Brake service": 32, "Tyre replacement": 28,
                       "Chain and drive": 40, "Damage repair": 120, "Battery service": 95}
SCORE_P_NORMAL = [0.02, 0.05, 0.18, 0.40, 0.35]
SCORE_P_DAMAGE = [0.25, 0.35, 0.25, 0.10, 0.05]

maint_rows = []
for (bid, bt, _, _, _, _), bought in zip(bikes_rows, bike_purchase):
    d = bought + timedelta(days=int(rng.normal(70, 15)))
    while d <= END:
        p = SERVICE_P.copy()
        if bt != "Electric":
            p[5] = 0.0
        p = p / p.sum()
        stype = str(rng.choice(SERVICE_TYPES, p=p))
        cost = round(float(rng.lognormal(math.log(SERVICE_COST_MEDIAN[stype]), 0.25)), 2)
        score = int(rng.choice([1, 2, 3, 4, 5], p=SCORE_P_DAMAGE if stype == "Damage repair" else SCORE_P_NORMAL))
        maint_rows.append([bid, d.isoformat(), stype, score, cost])
        d += timedelta(days=max(30, int(rng.normal(85, 18))))   # strictly later date, so (bike_id, date) stays unique
print(len(bikes_rows), "bikes,", len(maint_rows), "maintenance rows")

320 bikes, 3357 maintenance rows


### 4.4 Members

In [8]:
EMAIL_DOMAINS = ["gmail.com", "outlook.com", "hotmail.co.uk", "yahoo.co.uk", "icloud.com", "herts.ac.uk"]
EMAIL_P = [0.50, 0.15, 0.12, 0.09, 0.07, 0.07]


def clean(s):
    return s.lower().replace("'", "").replace(" ", "")


def make_email(first, last, birth_year):
    f, l = clean(first), clean(last)
    yy = (birth_year or int(rng.integers(1970, 2005))) % 100
    pattern = rng.choice(["f.l", "fl", "f_l", "i.l", "fl_yy", "f.lnn"], p=[0.35, 0.15, 0.1, 0.1, 0.15, 0.15])
    local = {
        "f.l": f"{f}.{l}", "fl": f"{f}{l}", "f_l": f"{f}_{l}", "i.l": f"{f[0]}.{l}",
        "fl_yy": f"{f}{l}{yy:02d}", "f.lnn": f"{f}.{l}{int(rng.integers(1, 99))}",
    }[str(pattern)]
    return f"{local}@{rng.choice(EMAIL_DOMAINS, p=EMAIL_P)}"


def make_phone():
    digits = "".join(str(int(x)) for x in rng.integers(0, 10, size=9))
    return f"07{digits[:3]} {digits[3:]}"


TIERS = ["Casual", "Standard", "Premium"]
TIER_P = [0.45, 0.35, 0.20]
PAY_OPTIONS = ["Debit card", "Credit card", "Apple Pay", "Google Pay", "Direct debit"]
PAY_P_BY_TIER = {                              # casual riders cannot pay by direct debit; premium mostly do
    "Casual":   [0.45, 0.20, 0.20, 0.15, 0.00],
    "Standard": [0.38, 0.14, 0.20, 0.16, 0.12],
    "Premium":  [0.10, 0.05, 0.10, 0.10, 0.65],
}

members = []
for mid in range(1, N_BASE_MEMBERS + 1):
    first, last = str(rng.choice(FIRST_NAMES)), str(rng.choice(LAST_NAMES))
    by = int(np.clip(rng.normal(1992, 13), 1950, 2008))
    tier = str(rng.choice(TIERS, p=TIER_P))
    join = date(2023, 1, 1) + timedelta(days=int((END - date(2023, 1, 1)).days * rng.beta(1.6, 1.0)))
    join = min(join, END - timedelta(days=7))
    members.append({
        "member_id": mid,
        "full_name": f"{first} {last}",
        "email": make_email(first, last, by),
        "phone": make_phone(),
        "birth_year": by,
        "home_town": str(rng.choice(TOWN_NAMES)),
        "membership_tier": tier,
        "preferred_payment": str(rng.choice(PAY_OPTIONS, p=PAY_P_BY_TIER[tier])),
        "join_date": join,
    })

## 5. Injecting missing and duplicate data

In [9]:
for m in members:
    if rng.random() < RATES["email_missing"]:
        m["email"] = None
    if rng.random() < RATES["phone_missing"]:
        m["phone"] = None
    if rng.random() < RATES["birth_year_missing"]:
        m["birth_year"] = None


def messy_name(name):
    first, last = name.split(" ", 1)
    r = rng.random()
    if r < 0.50:
        return name
    if r < 0.65:
        return name.lower()
    if r < 0.75:
        return f"{last}, {first}"
    if len(first) > 3:
        k = int(rng.integers(1, len(first) - 2))
        first = first[:k] + first[k + 1] + first[k] + first[k + 2:]
    return f"{first} {last}"


n_dup = int(round(RATES["member_duplicate"] * N_BASE_MEMBERS))
originals = rng.choice(N_BASE_MEMBERS, size=n_dup, replace=False)
dup_log = []
next_id = N_BASE_MEMBERS + 1
for oi in originals:
    o = members[int(oi)]
    d = dict(o)
    d["member_id"] = next_id
    d["full_name"] = messy_name(o["full_name"])
    d["join_date"] = min(o["join_date"] + timedelta(days=int(rng.integers(20, 400))), END - timedelta(days=1))
    if o["phone"] and rng.random() < 0.5:
        d["phone"] = o["phone"].replace(" ", "")
    if rng.random() < 0.3:
        d["membership_tier"] = str(rng.choice(TIERS, p=TIER_P))
    members.append(d)
    dup_log.append((o["member_id"], next_id, o["full_name"], d["full_name"]))
    next_id += 1

members_df = pd.DataFrame(members)
print(f"{len(members_df)} member rows, of which {n_dup} are injected duplicate people")
pd.DataFrame(dup_log, columns=["original_id", "duplicate_id", "original_name", "duplicate_name"]).head(8)

1456 member rows, of which 56 are injected duplicate people


,original_id,duplicate_id,original_name,duplicate_name
0,911,1401,Imogen Webb,imogen webb
1,652,1402,Matthew Webb,Matthew Webb
2,240,1403,Mohammed Nwosu,Mohammed Nwosu
3,1206,1404,Zara Campbell,zara campbell
4,398,1405,Siobhan Parker,Siobhan Parker
5,568,1406,Aarav Jenkins,Aarav Jenkins
6,979,1407,Natalie Harrison,natalie harrison
7,26,1408,Chidi Graham,Chdii Graham


## 6. Trips

In [10]:
start_idx = {d: i for i, d in enumerate(days)}

tier_mult = {"Casual": 0.5, "Standard": 1.0, "Premium": 1.6}
w_member = rng.lognormal(0, 1.0, size=len(members_df)) * members_df["membership_tier"].map(tier_mult).to_numpy()
join_idx = np.array([max(0, (j - START).days) for j in members_df["join_date"]])

w_bike = rng.lognormal(0, 0.4, size=N_BIKES) * np.where(bike_types == "Electric", 1.3, 1.0)
buy_idx = np.array([(b - START).days for b in bike_purchase])

day_ids = np.arange(N_DAYS)
cum_member = np.cumsum(w_member[None, :] * (join_idx[None, :] <= day_ids[:, None]), axis=1)
cum_bike = np.cumsum(w_bike[None, :] * (buy_idx[None, :] <= day_ids[:, None]), axis=1)

is_weekend = np.array([d.weekday() >= 5 for d in days])
day_w = np.exp(0.05 * (temp - 10)) * np.where(rain > 2, 0.55, 1.0) * np.where(is_weekend, 1.12, 1.0)
day_w = day_w / day_w.sum(axis=1, keepdims=True)

start_station = rng.choice(n_stations, size=N_TRIPS, p=station_pop)
start_day = np.zeros(N_TRIPS, dtype=int)
for ti in range(len(TOWN_NAMES)):
    sel = np.where(station_town_idx[start_station] == ti)[0]
    start_day[sel] = rng.choice(N_DAYS, size=len(sel), p=day_w[ti])


def sample_hour(weekend):
    if weekend:
        h = rng.normal(13.5, 3.3)
    else:
        u = rng.random()
        h = rng.normal(8.25, 0.9) if u < 0.32 else rng.normal(17.5, 1.1) if u < 0.64 else rng.uniform(9, 21)
    return float(np.clip(h, 5.5, 23.4))


start_dt = []
for di in start_day:
    midnight = datetime.combine(days[int(di)], datetime.min.time())
    start_dt.append(midnight + timedelta(hours=sample_hour(is_weekend[int(di)]), seconds=int(rng.integers(0, 60))))
start_dt = np.array(start_dt)
order = np.argsort(start_dt)
start_station, start_day, start_dt = start_station[order], start_day[order], start_dt[order]

centre = np.array([[TOWNS[t][0], TOWNS[t][1]] for t in TOWN_NAMES])
neighbours = []
for ti in range(len(TOWN_NAMES)):
    dist = [haversine_km(*centre[ti], *centre[tj]) if tj != ti else 1e9 for tj in range(len(TOWN_NAMES))]
    neighbours.append(list(np.argsort(dist)[:2]))
stations_in_town = [np.where(station_town_idx == ti)[0] for ti in range(len(TOWN_NAMES))]

SPEED = {"Standard": (13.0, 1.8), "Electric": (17.5, 1.8), "Cargo": (10.0, 1.4)}


def fare(tier, bike_type, minutes):
    """Tier-specific pricing. Casual pays an unlock fee plus per-minute; higher tiers get free minutes."""
    ebike_extra = {"Casual": 0.10, "Standard": 0.05, "Premium": 0.03}[tier] if bike_type == "Electric" else 0.0
    cargo_extra = 0.05 if bike_type == "Cargo" else 0.0
    if tier == "Casual":
        f = 1.00 + (0.15 + cargo_extra) * minutes + ebike_extra * minutes
    elif tier == "Standard":
        f = 0.10 * max(0, minutes - 30) + (ebike_extra + cargo_extra) * minutes
    else:
        f = 0.05 * max(0, minutes - 60) + (ebike_extra + cargo_extra) * minutes
    return round(min(f, 20.0), 2)


RATING_BASE = np.array([0.04, 0.06, 0.14, 0.34, 0.42])
RATING_WET = np.array([1.6, 1.4, 1.1, 0.9, 0.7])

tier_of = members_df["membership_tier"].to_numpy()
ids_of = members_df["member_id"].to_numpy()

trips = []
for k in range(N_TRIPS):
    di = int(start_day[k])
    s = int(start_station[k])
    ti = int(station_town_idx[s])

    mi = int(np.searchsorted(cum_member[di], rng.random() * cum_member[di, -1], side="right"))
    bi = int(np.searchsorted(cum_bike[di], rng.random() * cum_bike[di, -1], side="right"))
    tier, btype = tier_of[mi], bike_types[bi]

    r = rng.random()
    if r < 0.08:
        e = s
    elif r < 0.16:
        tj = int(rng.choice(neighbours[ti]))
        pool = stations_in_town[tj]
        e = int(rng.choice(pool))
    else:
        pool = [x for x in stations_in_town[ti] if x != s]
        e = int(rng.choice(pool))

    if e == s:
        dist_km = float(rng.lognormal(math.log(2.5), 0.5))
    else:
        dist_km = 1.3 * haversine_km(station_lat[s], station_lon[s], station_lat[e], station_lon[e])
        dist_km *= float(rng.normal(1.0, 0.08))
    dist_km = max(dist_km, 0.4)

    mu, sd = SPEED[btype]
    speed = max(6.0, float(rng.normal(mu, sd)))
    minutes = int(max(1, round(dist_km / speed * 60 + rng.exponential(1.5))))
    minutes = min(minutes, 1440)

    t0 = start_dt[k]
    t1 = t0 + timedelta(minutes=minutes)
    rain_mm = rain[ti, di]
    p = RATING_BASE * (RATING_WET if rain_mm > 5 else 1.0)
    rating = int(rng.choice([1, 2, 3, 4, 5], p=p / p.sum()))

    trips.append([
        0, int(ids_of[mi]), int(bi + 1),
        s + 1, int(rng.integers(1, station_cap[s] + 1)),
        e + 1, int(rng.integers(1, station_cap[e] + 1)),
        t0.strftime("%Y-%m-%d %H:%M:%S"), t1.strftime("%Y-%m-%d %H:%M:%S"),
        minutes, round(dist_km, 2), fare(tier, btype, minutes), rating,
    ])

Blank out some ratings and distances, then clone a few trips as duplicates.

In [11]:
RATING_I, DIST_I = 12, 10
for t in trips:
    if rng.random() < RATES["rating_missing"]:
        t[RATING_I] = None
    if rng.random() < RATES["distance_missing"]:
        t[DIST_I] = None

for i, t in enumerate(trips, start=1):
    t[0] = i

n_trip_dup = int(round(RATES["trip_duplicate"] * N_TRIPS))
dup_src = rng.choice(len(trips), size=n_trip_dup, replace=False)
trip_dup_log = []
nid = len(trips) + 1
for si in dup_src:
    clone = list(trips[int(si)])
    clone[0] = nid
    trips.append(clone)
    trip_dup_log.append((trips[int(si)][0], nid))
    nid += 1
print(len(trips), "trip rows, of which", n_trip_dup, "are injected duplicates")

weather_rows = [(d, t, tc, None if rng.random() < RATES["rainfall_missing"] else rm) for d, t, tc, rm in weather_full]
for row in maint_rows:
    if rng.random() < RATES["cost_missing"]:
        row[4] = None
maint_rows = [tuple(r) for r in maint_rows]

25375 trip rows, of which 375 are injected duplicates


## 7. Write to SQLite and validate

In [12]:
members_rows = [
    (int(m.member_id), m.full_name, m.email, m.phone,
     None if pd.isna(m.birth_year) else int(m.birth_year),
     m.home_town, m.membership_tier, m.preferred_payment, m.join_date.isoformat())
    for m in members_df.itertuples(index=False)
]

cur = con.cursor()
cur.executemany("INSERT INTO towns VALUES (?,?,?,?)", towns_rows)
cur.executemany("INSERT INTO stations VALUES (?,?,?,?,?,?,?)", stations_rows)
cur.executemany("INSERT INTO docks VALUES (?,?,?)", docks_rows)
cur.executemany("INSERT INTO bikes VALUES (?,?,?,?,?,?)", bikes_rows)
cur.executemany("INSERT INTO members VALUES (?,?,?,?,?,?,?,?,?)", members_rows)
cur.executemany("INSERT INTO weather_daily VALUES (?,?,?,?)", weather_rows)
cur.executemany("INSERT INTO maintenance_log VALUES (?,?,?,?,?)", maint_rows)
cur.executemany("INSERT INTO trips VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?)", [tuple(t) for t in trips])
con.commit()
print("inserted")

inserted


In [13]:
counts = {t: con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
          for t in ["towns", "stations", "docks", "bikes", "members", "weather_daily", "maintenance_log", "trips"]}
pd.Series(counts, name="rows").to_frame()

,rows
towns,8
stations,40
docks,836
bikes,320
members,1456
weather_daily,2920
maintenance_log,3357
trips,25375


In [14]:
print("foreign key violations:", con.execute("PRAGMA foreign_key_check").fetchall())
print("integrity check:", con.execute("PRAGMA integrity_check").fetchone()[0])

foreign key violations: []
integrity check: ok


In [15]:
checks = {
    "members.email": ("members", "email"), "members.phone": ("members", "phone"),
    "members.birth_year": ("members", "birth_year"), "trips.rating": ("trips", "rating"),
    "trips.distance_km": ("trips", "distance_km"), "weather_daily.rainfall_mm": ("weather_daily", "rainfall_mm"),
    "maintenance_log.cost_gbp": ("maintenance_log", "cost_gbp"),
}
rows = []
for label, (tbl, col) in checks.items():
    n, nulls = con.execute(f"SELECT COUNT(*), SUM({col} IS NULL) FROM {tbl}").fetchone()
    rows.append((label, nulls, n, round(100 * nulls / n, 1)))
pd.DataFrame(rows, columns=["column", "nulls", "rows", "pct_null"])

,column,nulls,rows,pct_null
0,members.email,82,1456,5.6
1,members.phone,180,1456,12.4
2,members.birth_year,70,1456,4.8
3,trips.rating,10282,25375,40.5
4,trips.distance_km,753,25375,3.0
5,weather_daily.rainfall_mm,66,2920,2.3
6,maintenance_log.cost_gbp,129,3357,3.8


In [16]:
dup_members = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT lower(email) FROM members WHERE email IS NOT NULL
        GROUP BY lower(email) HAVING COUNT(*) > 1)
""").fetchone()[0]
dup_trips = con.execute("""
    SELECT COUNT(*) FROM (
        SELECT member_id, bike_id, start_time FROM trips
        GROUP BY member_id, bike_id, start_time HAVING COUNT(*) > 1)
""").fetchone()[0]
print("email addresses shared by more than one member:", dup_members, "(injected duplicates:", n_dup, ")")
print("trips recorded more than once:", dup_trips, "(injected:", n_trip_dup, ")")

email addresses shared by more than one member: 60 (injected duplicates: 56 )
trips recorded more than once: 375 (injected: 375 )


In [17]:
pd.read_sql_query("""
    SELECT CASE WHEN rainfall_mm IS NULL THEN 'unknown'
                WHEN rainfall_mm > 2 THEN 'wet (>2 mm)' ELSE 'dry' END AS day_type,
           COUNT(*) AS trips,
           ROUND(AVG(avg_temp_c), 1) AS avg_temp,
           ROUND(AVG(duration_min), 1) AS avg_duration_min,
           ROUND(AVG(rating), 2) AS avg_rating
    FROM v_trip_weather GROUP BY day_type
""", con)

,day_type,trips,avg_temp,avg_duration_min,avg_rating
0,dry,22192,12.7,13.0,4.04
1,unknown,577,12.3,13.9,4.05
2,wet (>2 mm),2606,12.8,13.2,3.92


In [18]:
pd.read_sql_query("""
    SELECT m.membership_tier, COUNT(*) AS trips, ROUND(AVG(t.fare_gbp), 2) AS avg_fare,
           ROUND(AVG(t.duration_min), 1) AS avg_min
    FROM trips t JOIN members m ON m.member_id = t.member_id
    GROUP BY m.membership_tier ORDER BY CASE m.membership_tier WHEN 'Casual' THEN 1 WHEN 'Standard' THEN 2 ELSE 3 END
""", con)

,membership_tier,trips,avg_fare,avg_min
0,Casual,7127,3.39,12.8
1,Standard,10421,0.44,13.2
2,Premium,7827,0.18,13.0


In [19]:
try:
    con.execute("INSERT INTO trips VALUES (999999, 1, 1, 1, 1, 1, 1, '2026-01-01 10:00:00', '2026-01-01 09:00:00', 5, 1.0, 0.0, 3)")
except sqlite3.IntegrityError as e:
    print("rejected as expected:", e)
con.close()
print("done:", DB_PATH, f"({os.path.getsize(DB_PATH) / 1e6:.1f} MB)")

rejected as expected: CHECK constraint failed: end_time > start_time
done: hertsbike.db (4.1 MB)
